# Exercise: Working with Time-Series Data in Python

The city's bike-share program wants to forecast daily ride counts. They hand you a CSV export from their operations database — three years of daily data.

The data has problems: missing dates, duplicates, unit errors, outliers, and negative values. Your job is to find and fix them before any forecasting can happen.

## Data

`../data/bikeshare_rides.csv` — daily ride counts from January 2022 through December 2024 (with intentional errors)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

## Load and Inspect Raw Data

In [ ]:
raw = pd.read_csv("../data/bikeshare_rides.csv", parse_dates=["date"])

print(f"Raw data: {len(raw)} rows")
print(f"Date range: {raw['date'].min().date()} to {raw['date'].max().date()}")
expected_days = (raw["date"].max() - raw["date"].min()).days + 1
print(f"Expected rows for continuous daily data: {expected_days}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(raw["date"], raw["rides"], color="tab:red", linewidth=0.5)
ax.set_ylabel("Rides")
ax.set_title("Raw Bike-Share Data")
plt.tight_layout()
plt.show()

## Your Task

Implement the data cleaning and analysis functions below. Work through them in order — each builds on the previous step.

1. **`find_missing_dates`** — identify gaps in the date sequence
2. **`find_duplicates`** / **`fix_duplicates`** — detect and resolve duplicate dates
3. **`detect_unit_errors`** / **`fix_unit_errors`** — find and correct suspiciously low values
4. **`detect_outliers`** — flag values beyond 3 standard deviations
5. **`fix_negatives`** — handle negative ride counts
6. **`fill_gaps`** — interpolate missing dates to create a continuous daily series
7. **`compute_rolling_stats`** — calculate rolling mean and standard deviation
8. **`run_adf_test`** — perform the Augmented Dickey-Fuller stationarity test

In [ ]:
def find_missing_dates(df):
    """Return DatetimeIndex of dates missing from the sequence."""
    # TODO: implement
    raise NotImplementedError


def find_duplicates(df):
    """Return rows where the date appears more than once."""
    # TODO: implement
    raise NotImplementedError


def fix_duplicates(df):
    """Average ride counts for duplicate dates."""
    # TODO: implement
    raise NotImplementedError


def detect_unit_errors(df, expected_min=1000):
    """Return rows with suspiciously low positive values."""
    # TODO: implement
    raise NotImplementedError


def fix_unit_errors(df):
    """Multiply October 2023 hourly values by 24 to get daily totals."""
    # TODO: implement
    raise NotImplementedError


def detect_outliers(df, threshold=3):
    """Return rows where rides deviate more than `threshold` std from the mean."""
    # TODO: implement
    raise NotImplementedError


def fix_negatives(df):
    """Replace negative ride counts with NaN."""
    # TODO: implement
    raise NotImplementedError


def fill_gaps(df):
    """Set date as index, fill to daily frequency, interpolate missing values."""
    # TODO: implement
    raise NotImplementedError


def compute_rolling_stats(series, window=30):
    """Return (rolling_mean, rolling_std) for the given window."""
    # TODO: implement
    raise NotImplementedError


def run_adf_test(series):
    """Run ADF test and return dict with test_statistic, p_value, lags_used, critical_values."""
    # TODO: implement
    raise NotImplementedError

## Audit the Raw Data

In [ ]:
# Audit the raw data
missing = find_missing_dates(raw)
print(f"Missing dates ({len(missing)}):")
for d in missing:
    print(f"  {d.date()}")
print()

dupes = find_duplicates(raw)
print(f"Duplicate dates ({len(dupes)} rows):")
if len(dupes) > 0:
    print(dupes.to_string(index=False))
print()

low = detect_unit_errors(raw)
print(f"Suspiciously low values ({len(low)} rows)")
print()

outliers = detect_outliers(raw)
print(f"Outliers > 3 std ({len(outliers)} rows):")
for _, row in outliers.iterrows():
    print(f"  {row['date'].date()}: {row['rides']:.0f} rides")
print()

negs = raw[raw["rides"] < 0]
print(f"Negative values ({len(negs)} rows)")

## Clean the Data

In [ ]:
# Clean the data
df = fix_duplicates(raw)
df["date"] = pd.to_datetime(df["date"])
df = fix_unit_errors(df)
df = fix_negatives(df)
clean = fill_gaps(df)
print(f"Cleaned: {len(clean)} rows, {clean['rides'].isna().sum()} nulls remaining")

# Compare raw vs clean
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(raw["date"], raw["rides"], color="tab:red", linewidth=0.5)
axes[0].set_ylabel("Rides")
axes[0].set_title("Raw data")
axes[1].plot(clean.index, clean["rides"].values, color="black", linewidth=0.5)
axes[1].set_ylabel("Rides")
axes[1].set_title("Cleaned data")
plt.tight_layout()
plt.show()

## Stationarity Analysis

In [ ]:
# Stationarity analysis
adf = run_adf_test(clean["rides"])
print(f"ADF statistic: {adf['test_statistic']:.4f}")
print(f"p-value:       {adf['p_value']:.6f}")
print(f"Stationary: {'yes' if adf['p_value'] < 0.05 else 'no'}")
print()

diffed = clean["rides"].diff().dropna()
adf_diff = run_adf_test(diffed)
print(f"Differenced ADF statistic: {adf_diff['test_statistic']:.4f}")
print(f"Differenced p-value:       {adf_diff['p_value']:.6f}")
print(f"Stationary: {'yes' if adf_diff['p_value'] < 0.05 else 'no'}")

# Rolling stats
rmean, rstd = compute_rolling_stats(clean["rides"])
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(clean.index, clean["rides"].values, color="lightgray", linewidth=0.5)
axes[0].plot(rmean.index, rmean.values, color="black")
axes[0].set_ylabel("Rides")
axes[0].set_title("30-day rolling mean")
axes[1].plot(rstd.index, rstd.values, color="tab:orange")
axes[1].set_ylabel("Rides")
axes[1].set_title("30-day rolling std")
plt.tight_layout()
plt.show()

## Train/Test Split

In [ ]:
# Train/test split
train, test = clean["rides"].iloc[:-90], clean["rides"].iloc[-90:]
print(f"Train: {len(train)} days ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} days ({test.index[0].date()} to {test.index[-1].date()})")